# DAAN 545 — Clustering delay profiles

**Goal:** Group carrier–airport–months that have a similar delay shape, using two algorithms on the same scaled features.

**Input:** `data/processed/airline_delay_cause_with_airports.csv` (BTS delays + OpenFlights geography + hub fields). FAA staffing is not an input.

**Approach:**
1. Build delay rate and cause-share features. Drop months below the 10th percentile of `arr_flights`.
2. Standardize. Check PCA. Cluster in PC1–PC2 only if those two components explain at least 70%.
3. Fit **K-Means** and **Ward** hierarchical clustering. Choose k from the K-Means silhouette (k = 2…8).
4. Profile the clusters and compare the two labelings.

Write-up language and the numbers from the saved run: `docs/notebook_notes.md`.


## 0. Setup

Helpers live in `src/clustering.py` so the same feature build and k rule can be reused outside this notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.clustering import assignment_table, run_clustering, save_cluster_figures

DATA_PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
print(f"Project root: {ROOT}")


## 1. Fit both clusterings

K-Means and Ward see the same scaled matrix. k is the K-Means silhouette peak. Ward is cut at that same k so we can compare membership, not a different number of groups.


In [ ]:
panel = pd.read_csv(DATA_PROCESSED / "airline_delay_cause_with_airports.csv")
result = run_clustering(panel)

print("Rows in / clustered / dropped:",
      result["meta"]["n_rows_in"],
      result["meta"]["n_rows_clustered"],
      result["meta"]["n_rows_dropped"])
print(f"Flight cutoff (10th percentile): {result['meta']['min_arr_flights']:.0f}")
print("Features:", result["features"])
print(f"Dropped security share: {result['meta']['dropped_security_share']} "
      f"(std={result['meta']['security_share_std']})")
print(f"PC1–PC2 variance: {result['pca']['variance_first_two']:.1%}")
print(f"Cluster in PCA space: {result['pca']['cluster_in_pca_space']}")
print("k sweep:")
display(result["sweep"].round(3))
print("Chosen:", result["models"])


## 2. Profiles and agreement

Cause shares are means inside the cluster (delay minutes from that cause ÷ `arr_delay`). Slot % is the share of rows at JFK, LGA, EWR, or DCA. Those airports were not features.


In [ ]:
display(result["profiles"])
ct = pd.crosstab(
    result["frame"]["kmeans_cluster"],
    result["frame"]["ward_cluster"],
    rownames=["kmeans"],
    colnames=["ward"],
)
print("Crosstab (rows = K-Means, columns = Ward)")
display(ct)


## 3. Figures and exports

The PCA scatters are a 2D view of a fit that was **not** done in that plane (PC1–PC2 are under 70% of variance). Overlap in the picture matches a silhouette near 0.25.


In [ ]:
paths = save_cluster_figures(result, FIG_DIR)
for path in paths:
    print(path)

assignment_table(result).to_csv(DATA_PROCESSED / "cluster_assignments.csv", index=False)
result["sweep"].to_csv(DATA_PROCESSED / "cluster_k_sweep.csv", index=False)
result["profiles"].to_csv(DATA_PROCESSED / "cluster_profiles.csv", index=False)
ct.to_csv(DATA_PROCESSED / "cluster_method_crosstab.csv")
print("Saved cluster tables under data/processed/")
